# Copernicus DEM — quickstart (anonymous, no account)

Fetch a **Copernicus DEM GLO-30** tile over the Nile Delta from the public
AWS Open Data bucket, then read + plot it with `pyramids`. No credentials,
no SDK login, no API key — the bucket is anonymous. `earthlens.dem` just
hands you the raw COG; the cropping / mosaicking / hillshade is `pyramids`'
job.

Then a second bbox spanning two land tiles shows the multi-tile behaviour
and how to mosaic them into one continuous raster.

## Setup

`earthlens` provides the unified `EarthLens` entry point; `pyramids` reads the
downloaded COG; downloads go to a per-notebook temp directory.

In [ ]:
import tempfile
from pathlib import Path

from pyramids.dataset import Dataset

from earthlens.core import EarthLens
from earthlens.dem import Catalog

download_root = Path(tempfile.mkdtemp(prefix='dem-quickstart-'))
print(f'downloads will land under: {download_root}')

## The DEM catalog

Two datasets ship today — the ~30 m GLO-30 and the ~90 m GLO-90 grids.
Reading the catalog is offline.

In [ ]:
catalog = Catalog()
for dataset_id in sorted(catalog.datasets):
    row = catalog.get_dataset(dataset_id)
    print(
        f'{dataset_id:16} bucket={row.bucket:22} '
        f'token={row.resolution_token} ≈{row.native_resolution_m} m'
    )

## Download one GLO-30 tile over the Nile Delta

The Copernicus DEM grid is 1° x 1° tiles keyed by their SW corner. The bbox
here (`lat=[30.2, 30.8]`, `lon=[31.2, 31.8]`) lies entirely inside the tile
at `(30 N, 31 E)`, so one COG is returned.

In [ ]:
single_out = download_root / 'nile_delta'
paths = EarthLens(
    data_source='dem',
    dataset='cop-dem-glo-30',
    lat_lim=[30.2, 30.8],
    lon_lim=[31.2, 31.8],
    path=single_out,
).download()

for path in paths:
    print(f'  {path.name}  ({path.stat().st_size / 1024**2:.1f} MB)')

## Read the tile back with pyramids

The returned COG carries a WGS84 CRS and one elevation band (metres above
the EGM2008 geoid). Read it into a `pyramids.Dataset` and plot the raw
elevation.

In [ ]:
dem = Dataset.read_file(paths[0])
print(f'EPSG        : {dem.epsg}')
print(f'shape (y, x): {dem.rows} x {dem.columns}')
print(f'pixel size  : {dem.cell_size}')
print(f'no-data     : {dem.no_data_value}')

# pyramids reads the band, honours its declared no-data and places the raster on
# its own georeferenced axes, so there is no array to mask by hand and no extent
# to hard-code -- the previous extent was a guess at the tile's true bounds.
glyph = dem.plot(cmap='terrain', title='Copernicus DEM GLO-30 — Nile Delta tile')
glyph.cbar.set_label('elevation (m, EGM2008)')
glyph.ax.set_xlabel('longitude (°E)')
glyph.ax.set_ylabel('latitude (°N)')

## Crop the tile to the exact bbox

`earthlens` returns the whole 1° tile. To keep only the bbox pixels, hand
the raster to `pyramids.Dataset.crop` with the bbox in `[west, south, east,
north]` order.

In [ ]:
cropped = dem.crop(bbox=(31.2, 30.2, 31.8, 30.8), epsg=4326)
print(f'cropped shape (y, x): {cropped.rows} x {cropped.columns}')

glyph = cropped.plot(cmap='terrain', title='Copernicus DEM GLO-30 — cropped bbox')
glyph.cbar.set_label('elevation (m)')
glyph.ax.set_xlabel('longitude (°E)')
glyph.ax.set_ylabel('latitude (°N)')

## Multi-tile bbox and mosaic

A wider bbox spans several tiles — `earthlens` returns one COG per
intersected tile in row-major order. `pyramids.dataset.merge.merge_rasters`
composites them into one continuous raster.

The bbox below (`lat=[45.4, 45.6]`, `lon=[7.4, 8.6]`) intersects two
Alpine tiles at `(45 N, 7 E)` and `(45 N, 8 E)`.

In [ ]:
alps_out = download_root / 'alps'
alp_paths = EarthLens(
    data_source='dem',
    dataset='cop-dem-glo-90',
    lat_lim=[45.4, 45.6],
    lon_lim=[7.4, 8.6],
    path=alps_out,
).download()

for path in alp_paths:
    print(f'  {path.name}  ({path.stat().st_size / 1024**2:.2f} MB)')

print(f'{len(alp_paths)} tiles fetched')

In [ ]:
from pyramids.dataset.merge import merge_rasters

mosaic_path = alps_out / 'alps_mosaic.tif'
merge_rasters([str(p) for p in alp_paths], str(mosaic_path))

mosaic = Dataset.read_file(mosaic_path)
print(f'mosaic shape (y, x): {mosaic.rows} x {mosaic.columns}')
print(f'mosaic bbox        : {mosaic.bbox}')

glyph = mosaic.plot(
    cmap='terrain', title='Copernicus DEM GLO-90 — two-tile Alpine mosaic'
)
glyph.cbar.set_label('elevation (m, EGM2008)')
glyph.ax.set_xlabel('longitude (°E)')
glyph.ax.set_ylabel('latitude (°N)')

## What just happened — the account-free path

Every step above spoke to `s3://copernicus-dem-30m` and
`s3://copernicus-dem-90m` **anonymously**: no `~/.aws/credentials`, no
AWS profile, no environment variable, no signer. Copernicus DEM is the
only globally-consistent DEM reachable through `earthlens` without an
account, which is the entire justification for the `dem` backend.